In [6]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/BigContest_JSH/BigContest/data/transit')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import dask.dataframe as dd
import pandas as pd
import numpy as np
import time

import matplotlib.pyplot as plt


|축제 지역|행정동코드|
|---|---|
|대전|3020055000|
|부산|2635052000|
|강릉(스피드 스케이팅장)|5115057200|
|강릉(경포호수공원)|5115058000|
|임실|4575034000|
|서울|1156054000|

In [8]:
transit = pd.read_csv('transit.csv')
transit

,routeID,transitID,transit,description,time
0,SD01,1,WALK,-,303.0
1,SD01,2,TRAIN_1,KTX산천,4215.0
2,SD01,3,WALK,-,254.0
3,SD01,4,BUS,일반:707,1974.0
4,SD01,5,WALK,-,249.0
...,...,...,...,...,...
5820,II10,4,BUS_2,농어촌:임실-신흥촌,185.0
5821,II10,5,WALK,-,473.0
5822,II10,99,totalDistance,-,2991.0
5823,II10,99,totalTime,-,990.0


In [9]:
transit['origin_cd'] = transit.routeID.str[0]
transit['dest_cd'] = transit.routeID.apply(lambda x: x[1:3] if len(x) == 5 else x[1])
transit

,routeID,transitID,transit,description,time,origin_cd,dest_cd
0,SD01,1,WALK,-,303.0,S,D
1,SD01,2,TRAIN_1,KTX산천,4215.0,S,D
2,SD01,3,WALK,-,254.0,S,D
3,SD01,4,BUS,일반:707,1974.0,S,D
4,SD01,5,WALK,-,249.0,S,D
...,...,...,...,...,...,...,...
5820,II10,4,BUS_2,농어촌:임실-신흥촌,185.0,I,I
5821,II10,5,WALK,-,473.0,I,I
5822,II10,99,totalDistance,-,2991.0,I,I
5823,II10,99,totalTime,-,990.0,I,I


In [10]:
cd = {'S': '서울',
      'B': '부산 ',
      'D': '대전 ',
      'GA': '강릉(스피드 스케이트 경기장)',
      'GB': '강릉(경포호수광장)',
      'I': '임실 ',
}

transit.origin_cd = transit.origin_cd.map(cd)
transit.dest_cd = transit.dest_cd.map(cd)

In [11]:
transit.transit = transit.transit.apply(lambda x: x.split('_')[0])

In [12]:
cd_origin = transit['routeID'].str[:-2]
cd_origin = transit.copy()
cd_origin['routeID'] = cd_origin['routeID'].str[:-2]
cd_origin

,routeID,transitID,transit,description,time,origin_cd,dest_cd
0,SD,1,WALK,-,303.0,서울,대전
1,SD,2,TRAIN,KTX산천,4215.0,서울,대전
2,SD,3,WALK,-,254.0,서울,대전
3,SD,4,BUS,일반:707,1974.0,서울,대전
4,SD,5,WALK,-,249.0,서울,대전
...,...,...,...,...,...,...,...
5820,II,4,BUS,농어촌:임실-신흥촌,185.0,임실,임실
5821,II,5,WALK,-,473.0,임실,임실
5822,II,99,totalDistance,-,2991.0,임실,임실
5823,II,99,totalTime,-,990.0,임실,임실


In [13]:
cd_origin['routeID'].unique()

array(['SD', 'SB', 'SGA', 'SGB', 'SI', 'SS', 'BD', 'BB', 'BGA', 'BGB',
       'BI', 'BS', 'DD', 'GGA', 'GGB', 'II'], dtype=object)

# 최대 교통수단 수

In [14]:

filtered_transit = transit[(transit.transitID != 99) & (transit.transit != 'WALK')]

unique_transport_count = filtered_transit.pivot_table(
    index='dest_cd',
    values='transit',
    aggfunc=pd.Series.nunique
).reset_index()


unique_transport_count.columns = ['지역', '최대 교통수단 수']  # 컬럼명 변경

unique_transport_count

,지역,최대 교통수단 수
0,강릉(경포호수광장),4
1,강릉(스피드 스케이트 경기장),4
2,대전,3
3,부산,4
4,서울,3
5,임실,4


In [15]:
filtered_transit = cd_origin[(cd_origin.transitID != 99) & (cd_origin.transit != 'WALK')]

unique_transport_count = filtered_transit.pivot_table(
    index='routeID',
    values='transit',
    aggfunc=pd.Series.nunique
).reset_index()


unique_transport_count.columns = ['routeID', 'count']  # 컬럼명 변경

unique_transport_count

,routeID,count
0,BB,3
1,BD,3
2,BGA,3
3,BGB,3
4,BI,4
5,BS,3
6,DD,2
7,GGA,1
8,GGB,2
9,II,1


In [16]:
filtered_transit = cd_origin[(cd_origin.transitID != 99) & (cd_origin.transit != 'WALK')]

unique_transport_count = filtered_transit.groupby(['routeID','dest_cd'])['transit'].nunique().reset_index()

unique_transport_count.columns = ['루트별', '목적지', 'max transit']

unique_transport_count = unique_transport_count.drop_duplicates()

unique_transport_count

,루트별,목적지,max transit
0,BB,부산,3
1,BD,대전,3
2,BGA,강릉(스피드 스케이트 경기장),3
3,BGB,강릉(경포호수광장),3
4,BI,임실,4
5,BS,서울,3
6,DD,대전,2
7,GGA,강릉(스피드 스케이트 경기장),1
8,GGB,강릉(경포호수광장),2
9,II,임실,1


#환승 횟수

In [17]:
#도착지역별 평균 환승 횟수
df = cd_origin[(cd_origin.transitID != 99)].groupby('routeID').transitID.mean().reset_index()
df

,routeID,transitID
0,BB,3.013699
1,BD,4.298507
2,BGA,5.165605
3,BGB,5.165605
4,BI,5.705405
5,BS,3.867257
6,DD,2.958904
7,GGA,2.965909
8,GGB,3.037209
9,II,3.000000


In [18]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/1UhLsBGL7b8ZFfnss9E2TR7EXoS_NqtXCajelJgO3_6c#gid=0


/usr/local/lib/python3.10/dist-packages/google/colab/sheets.py:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return frame.applymap(_clean_val).replace({np.nan: None})


In [19]:
df = cd_origin[(cd_origin.transitID != 99)].groupby('dest_cd').transitID.mean().reset_index()
df

,dest_cd,transitID
0,강릉(경포호수광장),5.027273
1,강릉(스피드 스케이트 경기장),4.238434
2,대전,4.177570
3,부산,3.887781
4,서울,3.707273
5,임실,4.945591


In [20]:
# 각 routeID별 환승 횟수
def calculate_transfers(group):
    transfers = 0
    previous_transit = None

    for transit_type in group['transit']:
        if previous_transit is not None and previous_transit != transit_type:
            transfers += 1  # 교통수단이 변경되면 환승 횟수 증가
        previous_transit = transit_type

    return transfers

transfer_counts = filtered_transit.groupby('routeID').apply(calculate_transfers).reset_index()
transfer_counts.columns = ['루트별', '환승횟수']
transfer_counts

<ipython-input-20-13a8aad3acba>:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  transfer_counts = filtered_transit.groupby('routeID').apply(calculate_transfers).reset_index()


,루트별,환승횟수
0,BB,23
1,BD,244
2,BGA,101
3,BGB,101
4,BI,122
5,BS,175
6,DD,16
7,GGA,0
8,GGB,4
9,II,0


In [21]:
filtered_transit = cd_origin[(cd_origin['transitID'] != 99) & (cd_origin['transit'] != 'WALK')]
transfer_counts = filtered_transit.groupby(['routeID', 'dest_cd']).apply(calculate_transfers).reset_index()

transfer_counts.columns = ['루트별', '목적지', '환승횟수']

transfer_counts

<ipython-input-21-4cffb194d9fd>:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  transfer_counts = filtered_transit.groupby(['routeID', 'dest_cd']).apply(calculate_transfers).reset_index()


,루트별,목적지,환승횟수
0,BB,부산,23
1,BD,대전,244
2,BGA,강릉(스피드 스케이트 경기장),101
3,BGB,강릉(경포호수광장),101
4,BI,임실,122
5,BS,서울,175
6,DD,대전,16
7,GGA,강릉(스피드 스케이트 경기장),0
8,GGB,강릉(경포호수광장),4
9,II,임실,0


#마지막 도보시간

In [28]:
df=cd_origin[cd_origin.transit=='WALK'].groupby('routeID').last().reset_index()
df.time=df.time
df

,routeID,transitID,transit,description,time,origin_cd,dest_cd
0,BB,5,WALK,-,516.0,부산,부산
1,BD,9,WALK,-,303.0,부산,대전
2,BGA,11,WALK,-,1035.0,부산,강릉(스피드 스케이트 경기장)
3,BGB,11,WALK,-,312.0,부산,강릉(경포호수광장)
4,BI,11,WALK,-,1304.0,부산,임실
5,BS,7,WALK,-,219.0,부산,서울
6,DD,5,WALK,-,303.0,대전,대전
7,GGA,5,WALK,-,1035.0,None,강릉(스피드 스케이트 경기장)
8,GGB,7,WALK,-,245.0,None,강릉(경포호수광장)
9,II,5,WALK,-,473.0,임실,임실


In [30]:
median_time_per_route = cd_origin[cd_origin.transit == 'WALK'].groupby('routeID')['time'].median().reset_index()

# 도보시간을 분 단위로 변환
median_time_per_route['time'] = median_time_per_route['time']

# 결과 출력
median_time_per_route

,routeID,time
0,BB,198.0
1,BD,249.0
2,BGA,140.0
3,BGB,140.0
4,BI,227.0
5,BS,273.0
6,DD,230.5
7,GGA,186.0
8,GGB,186.0
9,II,118.0


In [33]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=median_time_per_route)

https://docs.google.com/spreadsheets/d/1YbwbFzcXLAYi-8GQom3nAs-FiiAjaeGruEok2SUM65E#gid=0


In [31]:
last_walk_time=df[['routeID', 'time']]
last_walk_time.columns=['경로별', '마지막 도보시간']
last_walk_time

,경로별,마지막 도보시간
0,BB,516.0
1,BD,303.0
2,BGA,1035.0
3,BGB,312.0
4,BI,1304.0
5,BS,219.0
6,DD,303.0
7,GGA,1035.0
8,GGB,245.0
9,II,473.0


In [32]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=last_walk_time)

https://docs.google.com/spreadsheets/d/1RjnNe25EbA6_k_1h1jAtuOBJOWSWBMOkWTHlnhDIy70#gid=0


/usr/local/lib/python3.10/dist-packages/google/colab/sheets.py:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return frame.applymap(_clean_val).replace({np.nan: None})


In [26]:
max_last_walk_time = df.groupby('routeID')['time'].max().reset_index()

max_last_walk_time.columns = ['목적지', '최대 도보시간']
max_last_walk_time

,목적지,최대 도보시간
0,BB,8.600000
1,BD,5.050000
2,BGA,17.250000
3,BGB,5.200000
4,BI,21.733333
5,BS,3.650000
6,DD,5.050000
7,GGA,17.250000
8,GGB,4.083333
9,II,7.883333


#마지막 도보 평균시간

In [27]:
average_last_walk_time = df.groupby('dest_cd')['time'].mean().reset_index()
average_last_walk_time.columns = ['목적지', '평균 도보시간']
average_last_walk_time

,목적지,평균 도보시간
0,강릉(경포호수광장),4.455556
1,강릉(스피드 스케이트 경기장),21.166667
2,대전,4.750000
3,부산,9.333333
4,서울,4.400000
5,임실,12.500000
